# 04 - Image Test (Local File)

This notebook validates image reasoning for the same sentiment triage use case, using a local image as evidence after text checks pass.

Pre-requisite: run `03_text_test.ipynb` first.

---

## Step 1 – Load environment variables

In [ ]:
import base64
import json
import mimetypes
import os
from pathlib import Path

import requests
from azure.identity import DefaultAzureCredential
from dotenv import load_dotenv

load_dotenv(dotenv_path="../env/.env")

endpoint = os.environ["AZURE_AI_ENDPOINT"].rstrip("/")
deployment = os.environ["AZURE_AI_DEPLOYMENT"]
auth_mode = os.environ.get("AZURE_AUTH_MODE", "aad").lower()
if auth_mode != "aad":
    raise RuntimeError(f"Unsupported auth mode for this demo: {auth_mode}. Expected 'aad'.")

credential = DefaultAzureCredential()
credential.get_token("https://cognitiveservices.azure.com/.default")
credential_source = "DefaultAzureCredential"
image_path = Path(os.environ["LOCAL_IMAGE_PATH"]).expanduser()

if not image_path.exists():
    raise FileNotFoundError(f"Image file not found: {image_path}")

output_dir = Path("../outputs")
output_dir.mkdir(exist_ok=True)

print(f"Using image       : {image_path.resolve()}")
print(f"Credential source : {credential_source}")

## Step 2 – Create clients / connections

Initialise any SDK clients using the loaded environment variables.

In [ ]:
mime_type = mimetypes.guess_type(image_path.name)[0] or "image/png"
image_b64 = base64.b64encode(image_path.read_bytes()).decode("utf-8")
data_url = f"data:{mime_type};base64,{image_b64}"

messages = [
    {"role": "system", "content": "You are a visual assistant for customer sentiment triage."},
    {
        "role": "user",
        "content": [
            {"type": "text", "text": "Review this customer-uploaded photo and provide: (1) likely issue, (2) severity level low/medium/high, (3) whether this should be expedited."},
            {"type": "image_url", "image_url": {"url": data_url}},
        ],
    },
]

print("Prepared multimodal message payload")

## Step 3 – Baseline test

A minimal call to confirm the deployment is reachable.

In [ ]:
url = f"{endpoint}/openai/deployments/{deployment}/chat/completions?api-version=2024-10-21"
token = credential.get_token("https://cognitiveservices.azure.com/.default").token
response = requests.post(
    url,
    headers={"Authorization": f"Bearer {token}", "Content-Type": "application/json"},
    json={"messages": messages, "max_completion_tokens": 300},
    timeout=90,
)
if response.status_code >= 400:
    raise RuntimeError(f"Image test failed: {response.status_code} {response.text}")

body = response.json()
image_response = body["choices"][0]["message"]["content"]
print(image_response)

(output_dir / "04_image_test_result.json").write_text(json.dumps(body, indent=2), encoding="utf-8")
print("Saved outputs/04_image_test_result.json")

## Step 4 – Demo scenario

Implement the key scenario that validates the customer hypothesis.

In [ ]:
followup_messages = [
    {"role": "system", "content": "You are a strict triage validator."},
    {
        "role": "user",
        "content": [
            {"type": "text", "text": "Give a final triage decision from this image only: queue_priority (P1/P2/P3) and one-sentence rationale."},
            {"type": "image_url", "image_url": {"url": data_url}},
        ],
    },
]

token = credential.get_token("https://cognitiveservices.azure.com/.default").token
response_2 = requests.post(
    url,
    headers={"Authorization": f"Bearer {token}", "Content-Type": "application/json"},
    json={"messages": followup_messages, "max_completion_tokens": 120},
    timeout=90,
)
if response_2.status_code >= 400:
    raise RuntimeError(f"Follow-up image test failed: {response_2.status_code} {response_2.text}")

body_2 = response_2.json()
print(body_2["choices"][0]["message"]["content"])

(output_dir / "04_image_test_followup.json").write_text(json.dumps(body_2, indent=2), encoding="utf-8")
print("Saved outputs/04_image_test_followup.json")

---

## Image checkpoint complete

Next: run **05_audio_text.ipynb** for speech-in to text-out validation.